# Fruit and Vegetable Fresh/Rotten Classification CNN.

## Tasks.
- Download, unzip and prepare dataset.
- Preprocess dataset.
- Train and test model.
- Implement Grading as set out in the case study.
- Export model.

In [167]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split

### Checking if the dataset exists, if it does not then download it from custom S3 bucket and prepare it.
For simplicity the kaggle dataset was uploaded to an S3 bucket for ease-of-access.

In [170]:
DATASET_DIRECTORY = '../dataset/Fruit And Vegetable Diseases Dataset'
BATCH_SIZE = 32
IMAGE_SIZE = (244, 244)
EPOCHS = 50
RAN_SEED = 181181 # Arbitrary seed for random number generation.

# Set seeds for random number generators.
tf.random.set_seed(RAN_SEED)
np.random.seed(RAN_SEED)
random.seed(RAN_SEED)

# Checking if dataset is present, if not then download and extract.
def check_dataset():
    if not os.path.exists(DATASET_DIRECTORY):
        print('Dataset not found, checking directory...')
        if not os.path.exists('../dataset'): os.mkdir('../dataset')
        if not os.path.isfile('../dataset/archive.zip'):
            print('Downloading archive...')
            import boto3
            from botocore import UNSIGNED
            from botocore.config import Config
            bucket_name = 'aai-university-content'
            endpoint = 'https://s3.fr-par.scw.cloud'
            file_key = 'archive.zip'
            local_file = '../dataset/archive.zip'
            s3 = boto3.client('s3',endpoint_url=endpoint,config=Config(signature_version=UNSIGNED))
            s3.download_file(bucket_name, file_key, local_file)
            print('Archive download complete.')

        print('Archive found, extracting archive...')
        from zipfile import ZipFile
        with ZipFile("../dataset/archive.zip", 'r') as zip_ref:
            zip_ref.extractall(path="../dataset")
        print('Archive extraction complete.')
        print('Removing archive...')
        os.remove('../dataset/archive.zip')
        print('Archive removed.')
    print('Dataset found.')

check_dataset()

Dataset not found, checking directory...
Archive download complete.
Archive found, extracting archive...
Archive extraction complete.
Removing archive...
Archive removed.
Dataset found.


### Identify classes, images and labels from dataset.


In [ ]:
dataset_img_paths = []
dataset_labels = []

# Identify the class names.
dataset_classes = sorted([x for x in os.listdir(DATASET_DIRECTORY) if os.path.isdir(os.path.join(DATASET_DIRECTORY, x))])

# Iterate through files to identify images and labels.
for label, dataset_class in enumerate(dataset_classes):
    directory = os.path.join(DATASET_DIRECTORY, dataset_class)
    for file_identifier in os.listdir(directory):
        if file_identifier.lower().endswith(('.jpg', '.jpeg', '.png')):
            dataset_img_paths.append(os.path.join(directory, file_identifier))
            dataset_labels.append(label)

print("Classes loaded: " + str(len(dataset_classes)))
print("Image locations loaded: " + str(len(dataset_img_paths)))
print("Labels loaded: " + str(len(dataset_labels)))

# Create simple table for displaying the total amount of data in each class.
dataset_label_ints = pd.Series(dataset_labels).value_counts().sort_index()
dataset_distribution = pd.DataFrame({
    "Class": dataset_classes,
    "Amount": dataset_label_ints.values
})
print("\nClass Totals: \n" + str(dataset_distribution))

### Split the data into training, validation and testing.

Using an 80/10/10 split of training, validation and testing data.

In [ ]:
X_training, X_valTest, y_training, y_valTest = train_test_split(
    dataset_img_paths,
    dataset_labels,
    test_size=0.2,
    random_state=RAN_SEED
)

X_validation, X_testing, y_validation, y_testing = train_test_split(
    X_valTest,
    y_valTest,
    test_size=0.5,
    random_state=RAN_SEED
)

print(
    'Training Data:', str(len(X_training)),
    '\nValidation Data:', str(len(X_validation)),
    '\nTesting Data:', str(len(X_testing))
)

### Image loading and dataset creation functions

In [ ]:
def load_images(dataset_image_path, dataset_image_label):
    dataset_image = tf.io.read_file(dataset_image_path)
    dataset_image = tf.image.decode_image(dataset_image, expand_animations=False)
    dataset_image = tf.image.resize(dataset_image, IMAGE_SIZE)
    dataset_image = tf.image.convert_image_dtype(dataset_image, tf.float32)
    return dataset_image, dataset_image_label

def create_tensorflow_dataset(dataset_image_path, dataset_image_label, train):
    tf_dataset = tf.data.Dataset.from_tensor_slices((dataset_image_path, dataset_image_label))
    tf_dataset = tf_dataset.map(load_images, num_parallel_calls=tf.data.AUTOTUNE)
    if train: tf_dataset = tf_dataset.shuffle(1000, RAN_SEED)
    tf_dataset = tf_dataset.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return tf_dataset

### Use above functions to instantiate the training, validation and testing datasets.

In [ ]:
tf_training_dataset = create_tensorflow_dataset(X_training, y_training, True)
tf_validation_dataset = create_tensorflow_dataset(X_validation, y_validation,False)
tf_testing_dataset = create_tensorflow_dataset(X_testing, y_testing, False)